# Day 4: Comprehensive Analysis Exercise

## Scenario: Multi-Quarter Sales Analysis

You're a data analyst tasked with analyzing sales performance across multiple quarters. You have data from different sources that need to be integrated, cleaned, analyzed, and visualized.

**Duration**: 2 hours (with instructor support)

**Skills Applied:**
- Loading and merging data from multiple files
- Data cleaning and transformation
- Date/time analysis
- GroupBy and aggregation
- Pivot tables
- Visualization
- Exporting results to Excel

---

## Exercise Overview

You will:
1. Load transaction, customer, and product data
2. Clean and integrate the datasets
3. Perform time-based analysis
4. Create pivot tables for insights
5. Calculate business metrics
6. Generate visualizations
7. Export comprehensive Excel report

**Work at your own pace. Instructor will circulate to help!**

---
## Step 1: Load and Inspect the Data

You have three CSV files:
- `transactions.csv`: Transaction records
- `customers.csv`: Customer information
- `products.csv`: Product master data

In [ ]:
# Setup: Create Sample Data for Analysis
# This cell simulates loading data from CSV files
# In real projects, you would use: pd.read_csv("filename.csv")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Set random seed for reproducible results
# np.random.seed(42) ensures we get the same "random" data every time
np.random.seed(42)

# Generate date range for 2 years of daily data
# pd.date_range() creates a sequence of dates
# Syntax: pd.date_range(start, end, freq='D') where freq='D' = daily
dates = pd.date_range('2023-01-01', '2024-12-31', freq='D')
n_transactions = 500

# Create transactions DataFrame
# np.random.choice() randomly selects from a list
# np.random.randint() generates random integers
transactions = pd.DataFrame({
    'TransactionID': [f'TXN-{i:04d}' for i in range(1, n_transactions+1)],  # f'TXN-{i:04d}' = zero-padded 4-digit IDs
    'Date': np.random.choice(dates, n_transactions),
    'CustomerID': np.random.choice([f'CUST-{i:03d}' for i in range(1, 51)], n_transactions),
    'ProductID': np.random.choice([f'PROD-{i:03d}' for i in range(1, 11)], n_transactions),
    'Quantity': np.random.randint(1, 10, n_transactions),
    'Amount': np.random.randint(500, 5000, n_transactions)
})

# Create customer reference data
# p=[0.2, 0.5, 0.3] sets probability weights for each option
customers = pd.DataFrame({
    'CustomerID': [f'CUST-{i:03d}' for i in range(1, 51)],
    'CustomerName': [f'Customer {i}' for i in range(1, 51)],
    'Region': np.random.choice(['EMEA', 'AMER', 'APAC'], 50),
    'CustomerType': np.random.choice(['Premium', 'Standard', 'Basic'], 50, p=[0.2, 0.5, 0.3])
})

# Create product reference data with cost info for margin calculations
products = pd.DataFrame({
    'ProductID': [f'PROD-{i:03d}' for i in range(1, 11)],
    'ProductName': ['Suite A', 'Suite B', 'Analytics', 'Premium', 'Basic', 
                    'Enterprise', 'Starter', 'Pro', 'Ultimate', 'Lite'],
    'Category': np.random.choice(['Software', 'Services', 'Hardware'], 10),
    'Price': [199, 99, 299, 499, 49, 599, 79, 249, 799, 39],
    'Cost': [80, 45, 130, 220, 20, 260, 35, 110, 350, 18]
})

print("✓ Data loaded successfully!")
print(f"\nTransactions: {len(transactions)} rows")
print(f"Customers: {len(customers)} rows")
print(f"Products: {len(products)} rows")

### Task 1.1: Inspect Each Dataset

Look at the first few rows and data types of each dataset.

In [ ]:
# Task 1.1: Inspect Each Dataset
# ALWAYS start by understanding your data structure
# Key methods: .head(), .dtypes, .shape, .info()

print("TRANSACTIONS:")
# .head() shows first 5 rows - check column names and sample values
print(transactions.head())
print("\nData types:")
# .dtypes shows the data type of each column
# Check: Are dates actually datetime? Are numbers numeric?
print(transactions.dtypes)

# Inspect customers - look for the key column (CustomerID)
print("\n" + "="*50)
print("CUSTOMERS:")
print(customers.head())
print("\nData types:")
print(customers.dtypes)

# Inspect products - look for the key column (ProductID)
print("\n" + "="*50)
print("PRODUCTS:")
print(products.head())
print("\nData types:")
print(products.dtypes)

---
## Step 2: Clean and Prepare Data

### Task 2.1: Convert Date Column
Make sure the Date column is datetime type.

In [ ]:
# Task 2.1: Convert Date Column to Datetime
# CRITICAL: Date columns often load as strings (object type)
# Must convert to datetime for date-based operations

# pd.to_datetime() converts strings/objects to datetime
# Syntax: df['column'] = pd.to_datetime(df['column'])
transactions['Date'] = pd.to_datetime(transactions['Date'])

# Verify the conversion worked
# dtype should now be 'datetime64[ns]', not 'object'
print("Date column type:", transactions['Date'].dtype)
print("✓ Date conversion complete")

### Task 2.2: Check for Missing Values
Identify any missing data in all datasets.

In [ ]:
# Task 2.2: Check for Missing Values
# Real data almost always has missing values - detect them first!
# .isna() returns True/False for each cell
# .isna().sum() counts missing values per column

print("Missing values in transactions:")
print(transactions.isna().sum())

print("\nMissing values in customers:")
print(customers.isna().sum())

print("\nMissing values in products:")
print(products.isna().sum())

# If you find missing values, decide how to handle:
# - fillna(value) to replace with a default
# - dropna() to remove rows with missing data
# - For this exercise, our sample data has no missing values

---
## Step 3: Integrate the Data

### Task 3.1: Join Transactions with Customers

In [ ]:
# Task 3.1: Join Transactions with Customers
# Use merge() to combine DataFrames on a common key
# Syntax: df1.merge(df2, on='key_column', how='left')
# - on: the column(s) that match in both DataFrames
# - how='left': keep all rows from left DataFrame (transactions)

df = transactions.merge(customers, on='CustomerID', how='left')

# Verify: row count should stay the same (500)
# Column count should increase (added customer columns)
print(f"After customer join: {len(df)} rows")
print(df.head())

### Task 3.2: Join with Products

In [ ]:
# Task 3.2: Join with Products
# Chain another merge to add product information
# Same syntax: df.merge(products, on='ProductID', how='left')
# Now each transaction row has customer AND product details

df = df.merge(products, on='ProductID', how='left')

# Check results: should have all original rows + new columns
print(f"After product join: {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")
print("\n✓ Data integration complete!")

### Task 3.3: Calculate Additional Metrics

In [ ]:
# Task 3.3: Calculate Additional Metrics
# Create calculated columns for business analysis
# Syntax: df['new_column'] = df['col1'] operator df['col2']

# 1. TotalRevenue = Quantity * Unit Price (what customer pays)
df['TotalRevenue'] = df['Quantity'] * df['Price']

# 2. TotalCost = Quantity * Unit Cost (what it costs us)
df['TotalCost'] = df['Quantity'] * df['Cost']

# 3. Profit = Revenue - Cost (what we earn)
df['Profit'] = df['TotalRevenue'] - df['TotalCost']

# 4. ProfitMargin % = (Profit / Revenue) * 100
# Use .round(2) to limit to 2 decimal places
df['ProfitMargin%'] = (df['Profit'] / df['TotalRevenue'] * 100).round(2)

print("✓ Calculated columns added")
print(df[['TransactionID', 'TotalRevenue', 'Profit', 'ProfitMargin%']].head())

---
## Step 4: Time-Based Analysis

### Task 4.1: Extract Date Components

In [ ]:
# Task 4.1: Extract Date Components
# Use the .dt accessor to extract parts of datetime columns
# This enables grouping/filtering by year, quarter, month, etc.
# Syntax: df['Date'].dt.property

# Extract year (2023 or 2024)
df['Year'] = df['Date'].dt.year

# Extract quarter (1, 2, 3, or 4)
df['Quarter'] = df['Date'].dt.quarter

# Extract month number (1-12)
df['Month'] = df['Date'].dt.month

# Extract month name (January, February, etc.)
df['MonthName'] = df['Date'].dt.month_name()

print("✓ Date components extracted")
print(df[['Date', 'Year', 'Quarter', 'MonthName']].head())

### Task 4.2: Monthly Revenue Trend

In [ ]:
# Task 4.2: Monthly Revenue Trend
# Group by time period to see trends over time
# .dt.to_period('M') converts dates to year-month periods (2023-01, 2023-02, etc.)
# Syntax: df.groupby(df['Date'].dt.to_period('M'))['column'].sum()

monthly_revenue = df.groupby(df['Date'].dt.to_period('M'))['TotalRevenue'].sum()

print("Monthly revenue:")
print(monthly_revenue.head(12))

# This creates a time series we can plot to see trends
# Increasing trend? Seasonality? Spikes?

---
## Step 5: Business Intelligence Queries

### Task 5.1: Regional Performance

In [ ]:
# Task 5.1: Regional Performance Analysis
# Use groupby with .agg() to calculate multiple metrics at once
# Syntax: df.groupby('category').agg({'col1': 'func1', 'col2': 'func2'})

regional_performance = df.groupby('Region').agg({
    'TotalRevenue': 'sum',        # Total revenue per region
    'Profit': 'sum',              # Total profit per region
    'ProfitMargin%': 'mean',      # Average profit margin
    'TransactionID': 'count'      # Number of transactions
}).round(2)

# Rename columns for clarity
regional_performance.columns = ['Total_Revenue', 'Total_Profit', 'Avg_Margin%', 'Transactions']

print("Regional Performance:")
print(regional_performance)

# Identify best performing region using idxmax()
# idxmax() returns the index (region name) of the maximum value
print("\n✓ Best region:", regional_performance['Total_Revenue'].idxmax())

### Task 5.2: Product Analysis

In [ ]:
# Task 5.2: Product Analysis - Top Performers
# Group by ProductName to see which products drive revenue
# Use sort_values() to rank and .head() to get top N

product_performance = df.groupby('ProductName').agg({
    'TotalRevenue': 'sum',    # Total revenue per product
    'Profit': 'sum',          # Total profit per product
    'Quantity': 'sum',        # Total units sold
    'TransactionID': 'count'  # Number of transactions
}).round(2)

# Rename for clarity
product_performance.columns = ['Revenue', 'Profit', 'Units_Sold', 'Transactions']

# Sort by Revenue descending and get top 5
# Syntax: df.sort_values('column', ascending=False).head(n)
top_products = product_performance.sort_values('Revenue', ascending=False).head(5)

print("Top 5 Products:")
print(top_products)

### Task 5.3: Customer Segment Analysis

In [ ]:
# Task 5.3: Customer Segment Analysis
# Analyze by CustomerType (Premium, Standard, Basic)
# Nested aggregations: count transactions, count unique customers

segment_analysis = df.groupby('CustomerType').agg({
    'TotalRevenue': 'sum',                    # Total revenue per segment
    'Profit': 'sum',                          # Total profit per segment
    'TransactionID': ['count', 'nunique']     # Count transactions, unique transaction IDs
})

# Note: Using ['count', 'nunique'] creates multi-level column headers
# 'count' = total rows, 'nunique' = unique values

print("Customer Segment Analysis:")
print(segment_analysis)

---
## Step 6: Create Pivot Tables

### Task 6.1: Region × Product Revenue Pivot

In [ ]:
# Task 6.1: Region × Product Revenue Pivot Table
# Pivot tables summarize data in a cross-tabular format
# Syntax: pd.pivot_table(df, index=rows, columns=cols, values=value, aggfunc='sum')

region_product_pivot = df.pivot_table(
    index='Region',           # Row labels
    columns='ProductName',    # Column labels
    values='TotalRevenue',    # Values to aggregate
    aggfunc='sum',            # Aggregation function
    fill_value=0,             # Replace NaN with 0
    margins=True,             # Add row/column totals
    margins_name='Total'      # Label for totals row/column
).round(0)

print("Region × Product Revenue Pivot:")
print(region_product_pivot)

### Task 6.2: Quarterly Performance Pivot

In [ ]:
# Task 6.2: Quarterly Performance Pivot
# Create a Year-Quarter label for clearer time-based analysis
# Combine Year and Quarter into a single string column

# Create combined YearQuarter label (e.g., "2023 Q1", "2023 Q2")
# .astype(str) converts numbers to strings for concatenation
df['YearQuarter'] = df['Year'].astype(str) + ' Q' + df['Quarter'].astype(str)

# Create pivot: Time periods (rows) × Regions (columns)
quarterly_pivot = df.pivot_table(
    index='YearQuarter',      # Rows: each quarter
    columns='Region',         # Columns: each region
    values='TotalRevenue',    # Values: revenue
    aggfunc='sum',            # Sum up revenue
    margins=True              # Include totals
).round(0)

print("Quarterly Revenue by Region:")
print(quarterly_pivot)

---
## Step 7: Visualize Key Insights

### Task 7.1: Monthly Revenue Trend

In [ ]:
# Task 7.1: Monthly Revenue Trend Line Chart
# Visualize the monthly revenue trend over time
# Use .plot() method from pandas Series

# Plot the monthly revenue as a line chart
# kind='line' draws connected points over time
monthly_revenue.plot(
    kind='line',            # Line chart for trends
    figsize=(12, 6),        # Width x Height in inches
    marker='o',             # Add circle markers at data points
    linewidth=2             # Thicker line for visibility
)

# Add chart formatting
plt.title('Monthly Revenue Trend', fontsize=16, fontweight='bold')
plt.ylabel('Revenue (€)')
plt.xlabel('Month')
plt.grid(True, alpha=0.3)   # Light gridlines for readability
plt.tight_layout()          # Adjust spacing to prevent cutoff
plt.show()

### Task 7.2: Regional Performance Dashboard

Create 2×2 dashboard showing:
1. Revenue by Region (bar)
2. Profit by Region (bar)
3. Top 5 Products (horizontal bar)
4. Customer Type Distribution (bar)

In [ ]:
# Task 7.2: Regional Performance Dashboard (2×2 Grid)
# Create multiple charts in one figure using plt.subplots()
# Syntax: fig, axes = plt.subplots(rows, cols, figsize=(width, height))

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Chart 1 (top-left): Revenue by Region - Bar Chart
# axes[0, 0] refers to row 0, column 0
regional_performance['Total_Revenue'].plot(kind='bar', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Revenue by Region', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('Revenue (€)')
axes[0, 0].tick_params(axis='x', rotation=0)  # Keep labels horizontal

# Chart 2 (top-right): Profit by Region - Bar Chart
regional_performance['Total_Profit'].plot(kind='bar', ax=axes[0, 1], color='green')
axes[0, 1].set_title('Profit by Region', fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel('Profit (€)')
axes[0, 1].tick_params(axis='x', rotation=0)

# Chart 3 (bottom-left): Top Products - Horizontal Bar Chart
# kind='barh' creates horizontal bars (good for long labels)
top_products['Revenue'].plot(kind='barh', ax=axes[1, 0], color='coral')
axes[1, 0].set_title('Top 5 Products by Revenue', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Revenue (€)')

# Chart 4 (bottom-right): Revenue by Customer Type
# Access multi-level column from segment_analysis
segment_analysis[('TotalRevenue', 'sum')].plot(kind='bar', ax=axes[1, 1], color='purple')
axes[1, 1].set_title('Revenue by Customer Type', fontsize=14, fontweight='bold')
axes[1, 1].set_ylabel('Revenue (€)')
axes[1, 1].tick_params(axis='x', rotation=0)

# Add overall title and adjust layout
plt.suptitle('Sales Performance Dashboard', fontsize=18, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

---
## Step 8: Export Results to Excel

### Task 8.1: Create Multi-Sheet Report

In [ ]:
# Task 8.1: Export Multi-Sheet Excel Report
# Use pd.ExcelWriter as context manager to write multiple sheets
# Syntax: with pd.ExcelWriter('file.xlsx', engine='xlsxwriter') as writer:

with pd.ExcelWriter('sales_analysis_report.xlsx', engine='xlsxwriter') as writer:
    
    # Sheet 1: Summary - Key business metrics
    summary = pd.DataFrame({
        'Metric': ['Total Revenue', 'Total Profit', 'Avg Profit Margin %', 'Total Transactions'],
        'Value': [
            df['TotalRevenue'].sum(),
            df['Profit'].sum(),
            df['ProfitMargin%'].mean(),
            len(df)
        ]
    })
    summary.to_excel(writer, sheet_name='Summary', index=False)
    
    # Sheet 2: Regional performance breakdown
    regional_performance.to_excel(writer, sheet_name='Regional')
    
    # Sheet 3: Product performance analysis
    product_performance.to_excel(writer, sheet_name='Products')
    
    # Sheet 4: Region × Product pivot table
    region_product_pivot.to_excel(writer, sheet_name='RegionProductPivot')
    
    # Sheet 5: Transaction details (first 1000 rows)
    # Use .head(1000) to limit size for large datasets
    df.head(1000).to_excel(writer, sheet_name='Transactions', index=False)

print("✓ Excel report created: sales_analysis_report.xlsx")
print("\nSheets created:")
print("  1. Summary - Key metrics")
print("  2. Regional - Regional performance")
print("  3. Products - Product analysis")
print("  4. RegionProductPivot - Pivot table")
print("  5. Transactions - Transaction details")

---
## Summary & Reflection

**Congratulations!** You've completed a comprehensive data analysis project.

### Skills You Applied:
✓ Data loading and integration (joins)
✓ Data cleaning and preparation
✓ Date/time manipulation
✓ GroupBy and aggregation
✓ Pivot tables
✓ Data visualization
✓ Excel export

### Key Insights from Analysis:
- Revenue trends over time
- Top performing regions and products
- Customer segment profitability
- Seasonal patterns

### Next Steps:
- Can you identify seasonality in the data?
- Which products have best profit margins?
- Are there underperforming regions?
- What recommendations would you make?

**You're now ready to tackle real-world data analysis projects!** 🎉